In [14]:
import pandas as pd
import numpy as np

In [15]:
# Create sample dataset
np.random.seed(0)

data = {
    "OrderID": range(1, 101),
    "CustomerID": np.random.randint(1000, 1100, 100),
    "ProductCategory": np.random.choice(["Electronics", "Clothing", "Grocery"], 100),
    "SalesAmount": np.random.randint(100, 1000, 100).astype(float),
    "Region": np.random.choice(["North", "South", "East", "West"], 100),
    "OrderDate": pd.date_range(start="2023-01-01", periods=100)
}

df = pd.DataFrame(data)

In [16]:
# Introduce missing values (only Electronics + South)
mask = (df["ProductCategory"] == "Electronics") & (df["Region"] == "South")
df.loc[mask, "SalesAmount"] = np.nan

df.to_csv("retail_sales.csv", index=False)

In [24]:
#from google.colab import files
#files.download("retail_sales.csv")
df.head()

,OrderID,CustomerID,ProductCategory,SalesAmount,Region,OrderDate,Imputation_Method
0,1,1044,Clothing,939.0,West,2023-01-01,None
1,2,1047,Clothing,796.0,West,2023-01-02,None
2,3,1064,Clothing,252.0,North,2023-01-03,None
3,4,1067,Electronics,691.0,West,2023-01-04,None
4,5,1067,Clothing,141.0,East,2023-01-05,None


In [18]:
# Fills missing SalesAmount using the median of the same ProductCategory and Region
regional_median = df.groupby(["ProductCategory", "Region"])["SalesAmount"].median()

In [19]:
# If the median cannot be calculated, use the overall median of that ProductCategory
category_median = df.groupby("ProductCategory")["SalesAmount"].median()

In [20]:
# Create a new column Imputation_Method
df["Imputation_Method"] = None

condition = (
    (df["ProductCategory"] == "Electronics") &
    (df["Region"] == "South") &
    (df["SalesAmount"].isna())
)

regional_count = 0
category_count = 0

for idx in df[condition].index:
    cat = df.loc[idx, "ProductCategory"]
    reg = df.loc[idx, "Region"]

    if not pd.isna(regional_median.loc[(cat, reg)]):
        df.loc[idx, "SalesAmount"] = regional_median.loc[(cat, reg)]
        df.loc[idx, "Imputation_Method"] = "Regional_Median"
        regional_count += 1
    else:
        df.loc[idx, "SalesAmount"] = category_median.loc[cat]
        df.loc[idx, "Imputation_Method"] = "Category_Median"
        category_count += 1

In [21]:
# Save the cleaned file as cleaned_retail_sales.csv
df.to_csv("cleaned_retail_sales.csv", index=False)

In [25]:
#from google.colab import files
#files.download("cleaned_retail_sales.csv")
df.head()

,OrderID,CustomerID,ProductCategory,SalesAmount,Region,OrderDate,Imputation_Method
0,1,1044,Clothing,939.0,West,2023-01-01,None
1,2,1047,Clothing,796.0,West,2023-01-02,None
2,3,1064,Clothing,252.0,North,2023-01-03,None
3,4,1067,Electronics,691.0,West,2023-01-04,None
4,5,1067,Clothing,141.0,East,2023-01-05,None


In [23]:
# Print the count of rows filled by each method
print("Regional_Median:", regional_count)
print("Category_Median:", category_count)

Regional_Median: 0
Category_Median: 13
